# Noise identification and reduction in echocardiographic frames

This notebook has two goals:

1. **Identify** which type of noise/perturbation is present in a given ultrasound frame by inspecting the histogram of a homogeneous dark region — the "background ROI" — extracted from *inside* the sector.
2. **Reduce** the identified noise with a targeted classical filter.

We work on the five single-kernel perturbed frames produced by `noise_generation.ipynb`. Each file has been corrupted by exactly *one* of the following kernels:

| Kernel | Physical model |
|---|---|
| `gaussian_blur` | Anisotropic Gaussian PSF |
| `contrast` | Gamma + linear rescale (attenuates contrast) |
| `speckle` | Multiplicative-like signal-dependent noise |
| `gaussian_noise` | Additive white Gaussian noise |
| `salt_pepper` | Random impulses (0 or 1) inside the sector |

> All filters below are applied *inside* the ultrasound sector only; the dark background is left untouched to avoid boundary artefacts.

In [ ]:
!pip install pydicom

In [ ]:
from pathlib import Path

import os
import re

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.ndimage as ndi
from scipy.ndimage import binary_closing, uniform_filter
from scipy.spatial import ConvexHull, QhullError
from skimage import exposure, restoration, filters
from skimage.draw import polygon
from pydicom import dcmread

import logging

logger = logging.getLogger(__name__)

plt.rcParams["figure.dpi"] = 110

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the drive containing the images
    drive.mount('/content/drive')

    # List contents (IPython magic command)
    !ls

In [ ]:
if IN_COLAB:
    # Change the working to the current notebook's directory
    %cd /content/drive/MyDrive/
else:
    # Ask the user to fill in the base data folder path
    base_data_folder = input("Please enter the path to the base data folder path: ")
    os.chdir(base_data_folder)

## Loading the perturbed gallery

`noise_generation.ipynb` saved one PNG per kernel using the naming convention `perturbed_<stem>_moderate_<kernel_name>.png`. We enumerate them here and also load the matching clean CAMUS frame from its DICOM so we can verify — at the end of the identification exercise — that our diagnosis was correct.

In [ ]:
# Directory populated by noise_generation.ipynb
PERTURBED_DIR = Path("./lab-0-data/perturbed_images")
CLEAN_DICOM = Path("./lab-0-data/camus/0001.dcm")

assert (
    PERTURBED_DIR.exists()
), f"Run noise_generation.ipynb first — {PERTURBED_DIR} is missing."
assert CLEAN_DICOM.exists(), f"Missing reference frame {CLEAN_DICOM}."

# Filename pattern: perturbed_<stem>_moderate_<kernel>.png
_pattern = re.compile(r"^perturbed_.+_moderate_(?P<kernel>[a-z_]+)\.png$")

perturbed: dict[str, np.ndarray] = {}
for f in sorted(PERTURBED_DIR.glob("perturbed_*_moderate_*.png")):
    m = _pattern.match(f.name)
    if not m:
        continue
    # `plt.imread` on a greyscale PNG returns a float array in [0, 1]. RGBA PNGs
    # (matplotlib defaults to writing RGBA when a colormap is given) come back
    # as (H, W, 4); we collapse to a single channel by averaging RGB.
    img = plt.imread(f)
    if img.ndim == 3:
        img = img[..., :3].mean(axis=-1)
    perturbed[m.group("kernel")] = img.astype(np.float32)

print("Loaded kernels:", sorted(perturbed))
assert (
    len(perturbed) == 5
), "Expected 5 kernels; re-run noise_generation.ipynb to regenerate the gallery."

# Load the clean reference frame from the raw DICOM.
# Normalise to [0, 1] float32 to match the perturbed PNGs.
_clean_ds = dcmread(CLEAN_DICOM)
clean = _clean_ds.pixel_array.astype(np.float32)
clean = (clean - clean.min()) / (clean.max() - clean.min())
print(f"Clean reference shape: {clean.shape}")

In [ ]:
# Quick preview: clean reference + all five perturbed frames side by side.
# The clean frame comes from the raw DICOM (native resolution); the perturbed
# frames come from PNGs saved at the same resolution. `imshow`'s aspect
# handling doesn't care about the small resolution differences that may exist.
_kernel_order = [
    "gaussian_blur",
    "contrast",
    "speckle",
    "gaussian_noise",
    "salt_pepper",
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
axes = axes.ravel()

axes[0].imshow(clean, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Clean (reference)", fontsize=10)
axes[0].axis("off")
for ax, k in zip(axes[1:], _kernel_order):
    ax.imshow(perturbed[k], cmap="gray", vmin=0, vmax=1)
    ax.set_title(k, fontsize=10)
    ax.axis("off")

fig.suptitle("Perturbed gallery (loaded from disk)", fontsize=12, fontweight="bold")
plt.show()

### Computing the sector mask

Echocardiographic frames contain a bright ultrasound fan surrounded by a black
border. We compute a simple binary mask of the sector so that filters are applied
only where there is valid signal. The approach used here — threshold + largest
connected component + convex hull — mirrors the logic in `src/masking/`.


In [ ]:
_THRESHOLD = 0.01
_CLOSE_RADIUS = 3

In [ ]:
def _disk_structuring_element(radius: int) -> np.ndarray:
    """Generate a 2-D disk-shaped structuring element for morphology operations.

    A disk SE is the standard isotropic element for smoothing binary shapes; it
    is used below by ``binary_closing`` to bridge small gaps in the sector
    threshold without biasing the boundary in any direction.

    Parameters
    ----------
    radius : int
        Radius of the disk in pixels.

    Returns
    -------
    np.ndarray
        Boolean array of shape ``(2*radius + 1, 2*radius + 1)``; ``True``
        wherever ``x**2 + y**2 <= radius**2``.
    """
    # Sparse 1-D coordinate arrays that broadcast to a 2-D grid — cheaper than
    # np.meshgrid for what is essentially a boolean lookup.
    y, x = np.ogrid[-radius : radius + 1, -radius : radius + 1]
    # Standard circle equation: pixels inside the disk of the given radius.
    return (x**2 + y**2) <= radius**2

In [ ]:
def compute_sector_mask(image: np.ndarray) -> np.ndarray:
    """Compute a binary mask of the ultrasound sector (the bright fan).

    The mask is the convex hull of pixels above a small intensity threshold,
    after a morphological closing that bridges small gaps in the raw
    binarisation. The convex hull deliberately includes interior dark pixels
    (blood pools, chamber interiors) so the mask covers the whole fan, not
    just its bright rim.

    Parameters
    ----------
    image : np.ndarray
        2-D image in ``[0, 1]``.

    Returns
    -------
    np.ndarray
        Boolean array of the same shape as ``image``; ``True`` inside the
        ultrasound fan.

    Raises
    ------
    ValueError
        If ``image`` is not 2-D.
    """
    if image.ndim != 2:
        raise ValueError(f"Expected 2-D image, got shape {image.shape}")

    # 1. Initial thresholding — rough outline of the fan, may have holes.
    binary = image > _THRESHOLD
    if not binary.any():
        logger.warning("compute_sector_mask: all-zero frame — returning empty mask")
        return np.zeros(image.shape, dtype=bool)

    # 2. Morphological closing — dilation followed by erosion. Bridges small
    #    gaps and smooths frayed edges without inflating the overall shape.
    se = _disk_structuring_element(_CLOSE_RADIUS)
    binary = binary_closing(binary, structure=se)

    # 3. Extract the (row, col) coordinates of every foreground pixel.
    rows, cols = np.nonzero(binary)  # pyright: ignore[reportAssignmentType]
    points = np.column_stack([rows, cols])

    # 4. Convex hull — the "rubber band" around all foreground pixels. Unifies
    #    disconnected blobs and fills interior dropouts.
    if len(points) < 3:
        # Not enough points to form a 2-D shape; fall back to the raw mask.
        return binary.astype(bool)
    try:
        hull = ConvexHull(points)
    except QhullError:
        # Degenerate (e.g. collinear) point set — fall back to the raw mask.
        return binary.astype(bool)

    # 5. Rasterise the hull polygon back into a boolean mask.
    hull_vertices = points[hull.vertices]
    rr, cc = polygon(hull_vertices[:, 0], hull_vertices[:, 1], shape=image.shape)
    mask = np.zeros(image.shape, dtype=bool)
    mask[rr, cc] = True
    return mask

In [ ]:
def show_comparison(
    original: np.ndarray,
    processed: np.ndarray,
    title_orig: str,
    title_proc: str,
    suptitle: str,
) -> None:
    """Plot a three-panel comparison: original, processed, absolute difference.

    Parameters
    ----------
    original : np.ndarray
        2-D frame in ``[0, 1]``, plotted on the left.
    processed : np.ndarray
        2-D frame in ``[0, 1]``, plotted in the middle.
    title_orig, title_proc : str
        Panel titles for the two frames.
    suptitle : str
        Bold figure-level title above all three panels.

    Returns
    -------
    None
        Renders the figure inline and returns nothing.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

    # Left: original frame — fixed grey colormap in [0, 1] for direct visual
    # comparison against the processed panel.
    axes[0].imshow(original, cmap="gray", vmin=0, vmax=1, aspect="equal")
    axes[0].set_title(title_orig, fontsize=9)
    axes[0].axis("off")

    # Middle: processed frame — same colormap and range.
    axes[1].imshow(processed, cmap="gray", vmin=0, vmax=1, aspect="equal")
    axes[1].set_title(title_proc, fontsize=9)
    axes[1].axis("off")

    # Right: pixel-wise absolute difference — highlights where the filter
    # acted. Auto-scaled by matplotlib so subtle changes remain visible.
    diff = np.abs(processed.astype(float) - original.astype(float))
    im = axes[2].imshow(diff, cmap="hot", aspect="equal")
    axes[2].set_title("Absolute difference", fontsize=9)
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(suptitle, fontsize=11, fontweight="bold")
    plt.show()

---

# Section 1 — Noise identification

Before you can *remove* noise you first have to *identify* it. The plan:

1. Extract a small homogeneous ROI from a dark region **inside** the ultrasound sector (typically the LV blood pool). Inside such a region the underlying anatomical signal is roughly constant, so any pixel-to-pixel variability we observe there is a direct read-out of the added noise.
2. Plot the histogram of that ROI.
3. Compare the shape of the histogram against the fingerprints listed below to decide which kernel produced the frame.

This approach is a standard quality assurance technique in medical imaging. By analyzing an area that should theoretically contain no structural details, you can isolate the noise profile without interference from the underlying anatomy.

## 1.1 An empirical dark-ROI extractor

The function below slides a square window over the frame, keeps only windows that are fully contained inside the sector, and returns the *darkest* such patch. Dark = "closest to zero" = "least anatomy" = "purest noise readout".

In [ ]:
def find_dark_homogeneous_roi(
    image: np.ndarray,
    sector_mask: np.ndarray,
    patch: int = 32,
) -> tuple[slice, slice]:
    """Return the row/column slices of the darkest fully-in-sector patch.

    We score candidate patches by *mean* intensity, not variance. A
    variance-based criterion would (incorrectly) prefer already-smoothed
    regions and hide the very noise we are trying to characterise — so we pick
    by darkness alone and let the student verify homogeneity visually via the
    ROI overlay.

    Parameters
    ----------
    image : np.ndarray
        2-D float32 frame in ``[0, 1]``.
    sector_mask : np.ndarray
        Boolean mask, ``True`` inside the ultrasound fan.
    patch : int, optional
        Side length of the square ROI in pixels (default 32).

    Returns
    -------
    tuple of slice
        ``(row_slice, col_slice)`` indexing the darkest patch fully contained
        in the sector.

    Raises
    ------
    ValueError
        If ``patch`` is not positive or larger than ``min(image.shape)``.
    RuntimeError
        If no candidate patch fits entirely inside the sector.
    """
    H, W = image.shape
    if patch <= 0 or patch > min(H, W):
        raise ValueError(f"patch={patch} out of range for image {H}x{W}")

    # Local mean of the image over a `patch x patch` window centred at every
    # pixel — computed in a single vectorised sweep by `uniform_filter`. The
    # value at (r, c) is the mean of the patch centred on (r, c).
    local_mean = uniform_filter(
        image.astype(np.float32), size=patch, mode="constant", cval=0.0
    )
    # Fraction of sector pixels in each candidate window; 1.0 == fully inside.
    local_mask_frac = uniform_filter(
        sector_mask.astype(np.float32), size=patch, mode="constant", cval=0.0
    )
    # Only consider windows entirely inside the sector; assign +inf to every
    # invalid centre so argmin skips them cleanly.
    valid = local_mask_frac >= 1.0 - 1e-6
    scores = np.where(valid, local_mean, np.inf)

    centre_r, centre_c = np.unravel_index(np.argmin(scores), scores.shape)
    if not np.isfinite(scores[centre_r, centre_c]):
        raise RuntimeError(
            "No fully-in-sector patch found — sector too small for patch size."
        )

    # Convert the centre coordinate back into slice indices, clamped to the
    # image bounds to guarantee a valid `patch x patch` view.
    half = patch // 2
    r = int(max(0, min(centre_r - half, H - patch)))
    c = int(max(0, min(centre_c - half, W - patch)))
    return slice(r, r + patch), slice(c, c + patch)

In [ ]:
# Recompute the sector mask of each perturbed frame independently. The
# perturbations may slightly bleed into the sector boundary, so we don't
# blindly reuse the clean-image mask.
sector_masks = {k: compute_sector_mask(img) for k, img in perturbed.items()}
clean_mask = compute_sector_mask(clean)

# In this pedagogical setting the anatomy is identical across the five frames,
# so the same ROI location works for all of them; we compute it once on the
# clean reference. In the real lab-0 exercise you would recompute the ROI on
# your perturbed frame — same call, same result up to a few pixels.
ROI_PATCH = 40
rr, cc = find_dark_homogeneous_roi(clean, clean_mask, patch=ROI_PATCH)
print(f"Selected dark ROI: rows {rr}, cols {cc}, size {ROI_PATCH}x{ROI_PATCH}")

# Clean-ROI histogram — the ground-truth reference the students won't have
# in the real lab; we use it as a visual anchor in the plot that follows.
clean_roi = clean[rr, cc]
hist_edges = np.linspace(0.0, 1.0, 65)
clean_hist, _ = np.histogram(clean_roi, bins=hist_edges)
print(
    f"Clean ROI: mean={clean_roi.mean():.3f}, std={clean_roi.std():.3f} "
    f"(baseline for the fingerprint check below)"
)

## 1.2 Histogram fingerprints — what to look for

Once you have the ROI, plot its intensity histogram (bins over `[0, 1]`). Each of the five perturbation kernels leaves a distinctive signature:

| Kernel | ROI histogram fingerprint | Auxiliary check |
|---|---|---|
| `gaussian_noise` | Symmetric bell around the ROI mean, **wider** than the clean-ROI histogram | Estimated std `≈ sqrt(10^(-PSNR/10))` |
| `speckle` | Also bell-shaped, but the width **scales with the local mean** | Repeat on a *bright* ROI — the width should grow visibly |
| `salt_pepper` | Bulk histogram + isolated **spikes at 0 and 1** | Fraction of extreme pixels ≈ `params.fraction` |
| `gaussian_blur` | **Narrower** than the clean-ROI histogram (variance suppressed by averaging); no added mass | Dark-ROI std collapses below the ~0.02 speckle floor of a clean frame |
| `contrast` | Same shape as the clean-ROI histogram but **shifted/compressed**; the global CDF is what changes | Whole-sector histogram is compressed towards the mean |

The next two cells run the ROI extraction, then plot each perturbed frame with its ROI drawn on top and its histogram side by side. In the real lab-0 exercise you won't have the clean image to overlay — but you *can* compare your dark-ROI histogram against a **bright-ROI** histogram taken from the *same* perturbed frame to decide between `gaussian_noise` and `speckle` (Section 1.3 below).

In [ ]:
# For each perturbed frame: show the frame with the ROI overlaid on the left,
# and the ROI histogram on the right (perturbed in red, clean reference in
# grey). In lab-0 you will *not* have the clean reference — the grey overlay
# is a teaching aid so you can calibrate your eye.
fig, axes = plt.subplots(
    len(_kernel_order),
    2,
    figsize=(12, 3.5 * len(_kernel_order)),
    constrained_layout=True,
)
for row, k in enumerate(_kernel_order):
    img = perturbed[k]
    roi = img[rr, cc]

    # Left: perturbed frame with ROI outlined in red.
    ax_img = axes[row, 0]
    ax_img.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax_img.add_patch(
        mpatches.Rectangle(
            (cc.start, rr.start),
            ROI_PATCH,
            ROI_PATCH,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
    )
    ax_img.set_title(f"{k} — dark ROI (red)", fontsize=10)
    ax_img.axis("off")

    # Right: overlaid histograms (clean in grey, perturbed in red).
    ax_hist = axes[row, 1]
    ax_hist.bar(
        hist_edges[:-1],
        clean_hist,
        width=np.diff(hist_edges),
        align="edge",
        color="lightgrey",
        label="clean ROI (reference)",
    )
    pert_hist, _ = np.histogram(roi, bins=hist_edges)
    ax_hist.bar(
        hist_edges[:-1],
        pert_hist,
        width=np.diff(hist_edges),
        align="edge",
        color="tab:red",
        alpha=0.6,
        label="perturbed ROI",
    )
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xlabel("Intensity")
    ax_hist.set_ylabel("Count")
    ax_hist.set_title(
        f"ROI histogram — mean={roi.mean():.3f}, std={roi.std():.3f}",
        fontsize=10,
    )
    ax_hist.legend(fontsize=8)

fig.suptitle(
    "Section 1 — Empirical noise fingerprints (dark ROI inside sector)",
    fontsize=12,
    fontweight="bold",
)
plt.show()

### 1.3 Dark ROI vs bright ROI — distinguishing `gaussian_noise` from `speckle`

Both `gaussian_noise` and `speckle` produce bell-shaped ROI histograms. The distinguishing property is that **speckle variance scales with the local mean** (`sqrt(x) * n` in our model), while additive Gaussian noise has the same variance everywhere.

To exploit this, we extract a second ROI — this time a **bright** homogeneous patch, e.g. from the myocardial wall — and compare the standard deviations of the dark-ROI and bright-ROI histograms.

In [ ]:
def find_bright_homogeneous_roi(
    image: np.ndarray,
    sector_mask: np.ndarray,
    patch: int = 32,
) -> tuple[slice, slice]:
    """Return the row/column slices of the brightest fully-in-sector patch.

    Mirror of :func:`find_dark_homogeneous_roi`, but scores by the *maximum*
    local mean. Used in §1.3 to compare dark- vs bright-ROI noise statistics —
    essential for telling apart additive Gaussian from multiplicative speckle.

    Parameters
    ----------
    image : np.ndarray
        2-D float32 frame in ``[0, 1]``.
    sector_mask : np.ndarray
        Boolean mask, ``True`` inside the ultrasound fan.
    patch : int, optional
        Side length of the square ROI in pixels (default 32).

    Returns
    -------
    tuple of slice
        ``(row_slice, col_slice)`` indexing the brightest patch fully contained
        in the sector.
    """
    H, W = image.shape
    # Same vectorised local mean as in the dark-ROI helper, but we now want
    # the maximum inside the sector instead of the minimum.
    local_mean = uniform_filter(
        image.astype(np.float32), size=patch, mode="constant", cval=0.0
    )
    local_mask_frac = uniform_filter(
        sector_mask.astype(np.float32), size=patch, mode="constant", cval=0.0
    )
    valid = local_mask_frac >= 1.0 - 1e-6
    # Invalid centres get -inf so argmax ignores them.
    scores = np.where(valid, local_mean, -np.inf)
    centre_r, centre_c = np.unravel_index(np.argmax(scores), scores.shape)
    half = patch // 2
    r = int(max(0, min(centre_r - half, H - patch)))
    c = int(max(0, min(centre_c - half, W - patch)))
    return slice(r, r + patch), slice(c, c + patch)


# --- Bright-vs-dark ROI test ------------------------------------------------
# Reuse the clean reference to locate a bright homogeneous patch — typically
# inside the myocardial wall.
brr, bcc = find_bright_homogeneous_roi(clean, clean_mask, patch=ROI_PATCH)

# Compare dark-ROI vs bright-ROI standard deviation for the two additive
# noise kernels:
#   * pure Gaussian noise -> variance is independent of the local mean, so
#     both stds should be similar (ratio ~ 1).
#   * speckle             -> variance scales with sqrt(mean), so the bright
#     ROI std should be noticeably larger (ratio > 1).
print(f"{'kernel':>16s} | dark std | bright std | ratio (bright/dark)")
print("-" * 60)
for k in ("gaussian_noise", "speckle"):
    dark_std = perturbed[k][rr, cc].std()
    bright_std = perturbed[k][brr, bcc].std()
    ratio = bright_std / max(dark_std, 1e-8)
    print(f"{k:>16s} | {dark_std:8.4f} | {bright_std:10.4f} | {ratio:.2f}")

# Overlay both ROIs on the speckle frame — the difference between dark and
# bright is most pronounced there.
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(perturbed["speckle"], cmap="gray", vmin=0, vmax=1)
ax.add_patch(
    mpatches.Rectangle(
        (cc.start, rr.start),
        ROI_PATCH,
        ROI_PATCH,
        linewidth=2,
        edgecolor="red",
        facecolor="none",
        label="dark ROI",
    )
)
ax.add_patch(
    mpatches.Rectangle(
        (bcc.start, brr.start),
        ROI_PATCH,
        ROI_PATCH,
        linewidth=2,
        edgecolor="lime",
        facecolor="none",
        label="bright ROI",
    )
)
ax.legend(loc="upper right")
ax.set_title("Dark ROI (red) vs bright ROI (green) on the speckle frame")
ax.axis("off")
plt.show()

### 1.4 Decision tree — putting it all together (reference-free)

Follow these checks in order on any perturbed frame; they are the exact mental steps you should run through in lab-0 when filling in the `perturbations` column of `answers.csv`. **None of them requires the clean reference** — each one calibrates against a known property of a healthy still-frame US image.

1. **Isolated spikes at 0 and 1 in the ROI histogram?** → `salt_pepper`.
2. **Symmetric bell around the ROI mean?** → additive noise. Compare a *dark* ROI to a *bright* ROI on the *same* frame:
   * bright-ROI std similar to (or smaller than) dark-ROI std → `gaussian_noise`
   * bright-ROI std visibly larger than dark-ROI std → `speckle`
3. **Dark ROI std < ~0.015?** → `gaussian_blur`. A clean US dark-ROI carries a granular *speckle floor* of std ~0.02–0.04; a blurred frame smooths that floor out. *Caveat*: this is a **local** test on a single 40×40 patch, so a badly-placed ROI (grazing the endocardium, or landing on already-smoothed tissue) can fool it. If the answer is borderline, complement it with the **Laplacian-variance sharpness metric from the theory lecture** — the variance of $\nabla^2 u$ inside the sector — which measures edge energy globally and is robust to ROI placement.
4. **Whole-sector std < ~0.10 (or P95–P5 range < ~0.4)?** → `contrast`. Every noise kernel *adds* energy to the sector std; blur leaves it essentially unchanged; only contrast attenuation *shrinks* the dynamic range.

> The thresholds above are calibrated on CAMUS 2CH / 4CH stills at native resolution. Different acquisition settings (different transducer, gain, depth) may shift them by 10–20 %, so treat them as ballpark cutoffs, not hard boundaries. A frame may of course carry *several* perturbations at once — the released lab-0 gallery mixes them; the checks above are independent, so tick each one that fires.

The next cell runs both checks on the single-kernel gallery and prints an automated verdict per frame — the "verdict" column is exactly what a student should conclude when only the perturbed image is available.

In [ ]:
# Reference-free diagnostics for the two "structural" perturbations that
# don't add mass to the ROI histogram: `gaussian_blur` and `contrast`.
#
# The trick is that each kernel violates a well-known property of a clean
# US frame that we can measure on the perturbed frame alone:
#
#   * `gaussian_blur` collapses the *speckle floor* everywhere. A clean US
#     dark-ROI carries a granular speckle texture with std ~0.02-0.04;
#     if the dark ROI is smoothed flat, blur is the culprit. NOTE: this is
#     a *local* test on a single 40x40 patch — a badly placed ROI can fool
#     it. Complement with the Laplacian-variance sharpness metric from the
#     theory lecture (variance of the Laplacian inside the sector — high on
#     sharp frames, low on blurred ones) for an independent global check.
#
#   * `contrast` compresses the *whole-sector dynamic range* symmetrically.
#     A clean US sector has std ~0.20-0.30 (P95-P5 ~0.6-0.9). Every noise
#     kernel *raises* sector std by adding energy; blur leaves it roughly
#     unchanged; only contrast attenuation shrinks it below ~0.10.


# Reference thresholds for a CAMUS-style still frame at native resolution.
# These are calibrated on the clean reference below, not on the perturbed
# gallery — students can reuse them on any similarly-acquired frame.
DARK_ROI_STD_THRESHOLD = 0.015  # below this -> blur
SECTOR_STD_THRESHOLD = 0.15  # below this -> contrast attenuation

print(f"{'frame':>16s} | dark ROI std | sector std | verdict")
print("-" * 70)
for k in _kernel_order + ["clean (ref)"]:
    if k == "clean (ref)":
        img_k, mask_k = clean, clean_mask
    else:
        img_k, mask_k = perturbed[k], sector_masks[k]

    roi_std = float(img_k[rr, cc].std())
    sec_std = float(img_k[mask_k].std())

    verdicts = []
    if roi_std < DARK_ROI_STD_THRESHOLD:
        verdicts.append("blur?")
    if sec_std < SECTOR_STD_THRESHOLD:
        verdicts.append("contrast?")
    verdict = ", ".join(verdicts) if verdicts else "-"

    print(f"{k:>16s} | {roi_std:12.4f} | {sec_std:10.3f} | {verdict}")

# Global histogram overlay — the same measurement, visualised. `contrast`
# collapses toward the sector mean; `gaussian_blur` looks almost identical
# to the clean reference in this view (it only changes local sharpness,
# not the intensity distribution).
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
edges = np.linspace(0, 1, 65)
ax.hist(
    clean[clean_mask].ravel(),
    bins=edges,
    color="lightgrey",
    label="clean (reference)",
    alpha=0.9,
)
ax.hist(
    perturbed["contrast"][sector_masks["contrast"]].ravel(),
    bins=edges,
    color="tab:orange",
    label="contrast",
    alpha=0.6,
)
ax.hist(
    perturbed["gaussian_blur"][sector_masks["gaussian_blur"]].ravel(),
    bins=edges,
    color="tab:blue",
    label="gaussian_blur",
    alpha=0.4,
)
ax.set_xlabel("Intensity")
ax.set_ylabel("Count (sector only)")
ax.set_title("Whole-sector histograms — spot the contrast compression")
ax.legend()
plt.show()

### 1.5 Calibrating thresholds from the released dataset

The two thresholds hard-coded above (`DARK_ROI_STD_THRESHOLD`, `SECTOR_STD_THRESHOLD`) were picked by inspecting the clean CAMUS frame `0001.dcm`. In the actual lab-0 release you will *not* have a clean reference — but you will have **many** perturbed frames (`answers.csv` has 20). Because each perturbation is sampled independently per frame, any given kernel affects only a *minority* of the gallery, so the **median** value of each statistic across the dataset is a good estimate of the "clean baseline" — no reference frame required.

Concretely:

1. Compute `sector_std` and `dark_roi_std` on every frame.
2. Take the **median** of each column across frames → the "typical clean" baseline.
3. Take the **median absolute deviation (MAD)** as a robust scale estimator (ignores the outliers caused by affected frames).
4. Threshold at `median − 3 · 1.4826 · MAD` (roughly a 2-sigma cut below the baseline). Any frame whose statistic falls below its threshold is a strong candidate for the corresponding kernel.

The cell below runs this procedure on our five-frame gallery and compares the data-driven thresholds against the hard-coded ones. **With only five frames — one per kernel — the demonstration is a caricature**: the "clean baseline" is contaminated by all five perturbations at once, so the data-driven cutoff will be *less* aggressive than the hand-tuned one. On the 20-frame `answers.csv` release the median converges to the actual clean baseline and the cutoffs sharpen up.

In [ ]:
# Collect the two reference-free statistics for every frame in the gallery.
metrics = ["sector_std", "dark_roi_std"]
stats: dict[str, dict[str, float]] = {}
for k in _kernel_order:
    img_k = perturbed[k]
    mask_k = sector_masks[k]
    stats[k] = {
        "sector_std": float(img_k[mask_k].std()),
        "dark_roi_std": float(img_k[rr, cc].std()),
    }

# Pretty-print the per-frame table.
print(f"{'frame':>16s} | " + " | ".join(f"{m:>13s}" for m in metrics))
print("-" * (18 + 16 * len(metrics)))
for k, s in stats.items():
    row = " | ".join(f"{s[m]:13.4f}" for m in metrics)
    print(f"{k:>16s} | {row}")


def robust_threshold_below_median(values: np.ndarray, k: float = 3.0) -> float:
    """Robust lower-tail cutoff: ``median - k * 1.4826 * MAD``.

    Used to derive reference-free thresholds from the whole gallery: as long
    as any single kernel affects only a *minority* of frames, the median of a
    per-frame statistic estimates the "clean baseline" and the MAD estimates
    its scale. The factor ``1.4826`` converts MAD to a Gaussian-consistent
    standard deviation; ``k=3`` corresponds to roughly a 2-sigma cut on the
    lower tail.

    Parameters
    ----------
    values : np.ndarray
        1-D array of per-frame statistics.
    k : float, optional
        Number of MAD-based standard deviations below the median (default 3.0).

    Returns
    -------
    float
        Threshold below which a frame is flagged as affected.
    """
    med = float(np.median(values))
    mad = float(np.median(np.abs(values - med)))
    return med - k * 1.4826 * mad


# Compute a data-driven cutoff for each statistic and compare it against
# the hand-tuned constants we've been using.
vectors = {m: np.array([stats[k][m] for k in _kernel_order]) for m in metrics}
auto_thresholds = {m: robust_threshold_below_median(vectors[m]) for m in metrics}
hardcoded = {
    "sector_std": SECTOR_STD_THRESHOLD,
    "dark_roi_std": DARK_ROI_STD_THRESHOLD,
}

print("\nThreshold comparison (statistic below threshold -> kernel present)")
print(f"{'statistic':>15s} | hard-coded | data-driven")
print("-" * 44)
for m in metrics:
    print(f"{m:>15s} | {hardcoded[m]:10.4f} | {auto_thresholds[m]:11.4f}")

# Strip plot: one point per frame per statistic, with both cutoffs overlaid.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, m in zip(axes, metrics):
    ax.scatter(vectors[m], np.arange(len(_kernel_order)), color="tab:blue", zorder=3)
    for idx, k in enumerate(_kernel_order):
        ax.annotate(
            k,
            (vectors[m][idx], idx),
            xytext=(6, 0),
            textcoords="offset points",
            fontsize=8,
            va="center",
        )
    ax.axvline(
        auto_thresholds[m],
        color="tab:red",
        linestyle="--",
        label=f"data-driven ({auto_thresholds[m]:.4f})",
    )
    ax.axvline(
        hardcoded[m],
        color="tab:green",
        linestyle=":",
        label=f"hard-coded ({hardcoded[m]:.4f})",
    )
    ax.set_yticks([])
    ax.set_xlabel(m)
    ax.legend(fontsize=8, loc="lower right")

fig.suptitle(
    "Section 1.5 — deriving reference-free thresholds from the gallery",
    fontsize=12,
    fontweight="bold",
)
plt.show()

---

# Section 2 — Noise reduction (one filter per identified noise type)

Now that we can identify each perturbation, we pair each one with a classical filter that specifically targets its statistical model. The mapping we'll use:

| Identified noise | Counter-filter | Rationale |
|---|---|---|
| `salt_pepper` | 3×3 **median** filter | The median is insensitive to extreme outliers — a single spike is simply outvoted. |
| `gaussian_noise` | **Bilateral** filter | Edge-aware smoothing: averages spatially close pixels *and* intensity-close pixels, so it suppresses AWGN without blurring the endocardium. |
| `speckle` | **Non-local means** (NLM) | Exploits self-similar texture — cardiac walls repeat across the image, and averaging over similar patches cancels the multiplicative noise. |
| `contrast` | **CLAHE** | Restores local contrast tile-by-tile with a clip limit that prevents runaway noise amplification. |
| `gaussian_blur` | **Unsharp mask** | Adds back a scaled high-pass of the image, restoring edge crispness lost to blurring. |

All filters below respect the sector mask — we run the filter on the whole frame and then restore the black background outside the sector to preserve the clean acquisition geometry.

## 2.1 `salt_pepper` → 3×3 median filter

**Model.** A fraction `f` of sector pixels is replaced by either `0` (pepper) or `1` (salt). The corrupted pixels are outliers relative to their local neighbourhood.

**Filter.** For each pixel, replace it with the median of its 3×3 neighbourhood:

$$\hat{u}(x, y) = \operatorname{median}\bigl\{ u(x', y') : (x', y') \in \mathcal{N}_{3\times 3}(x, y) \bigr\}$$

The median is a *rank statistic* — it ignores extreme values, unlike the mean which averages them in. That's why a single spike cannot bias the output.

**Main knob.** `size` (neighbourhood side). Larger → removes larger noise blobs but slightly blurs fine anatomy. 3×3 is enough for `fraction=0.02`.

In [ ]:
# --- Input ------------------------------------------------------------------
# Frame to be filtered and its sector mask.
img_sp = perturbed["salt_pepper"]
mask_sp = sector_masks["salt_pepper"]

# --- Filter -----------------------------------------------------------------
# A 3x3 median replaces every pixel with the median of its 8 neighbours + itself.
# Because the median is a rank statistic it ignores extreme values (0 or 1
# spikes), so a single impulse is simply out-voted by its neighbours. The 3x3
# window is the smallest that guarantees a strict majority even when two
# adjacent pixels are corrupted.
median_result = ndi.median_filter(img_sp, size=3).astype(np.float32)

# --- Mask restore -----------------------------------------------------------
# Reinstate the black background outside the sector (all photometric filters
# in Section 2 bleed slightly across the fan boundary).
median_result = np.where(mask_sp, median_result, 0.0)

show_comparison(
    img_sp,
    median_result,
    "salt_pepper (perturbed)",
    "median 3x3",
    "2.1 — Salt & pepper removal via median filtering",
)

## 2.2 `gaussian_noise` → bilateral filter

**Model.** Independent additive noise `y = x + n`, `n ~ N(0, σ²)`. Any linear low-pass smoothing will reduce the noise variance, but a plain Gaussian blur also blurs the endocardial edges we care about.

**Filter.** The **bilateral filter** combines a spatial Gaussian with an *intensity* Gaussian:

$$\hat{u}(p) = \frac{1}{Z(p)} \sum_{q \in \Omega} G_{\sigma_s}(\|p - q\|)\, G_{\sigma_r}(|u(p) - u(q)|)\, u(q)$$

The `G_{σ_r}` term downweights neighbours whose intensity differs strongly from the centre pixel — i.e. pixels *across an edge*. Result: within a homogeneous region the filter behaves like a Gaussian smoother; near an edge it collapses to averaging only same-side neighbours.

**Main knobs.**
- `sigma_spatial` — extent of the spatial neighbourhood (px). Larger → more smoothing.
- `sigma_color` — intensity tolerance. Small values preserve sharp edges; too small suppresses all denoising.

In [ ]:
# --- Input ------------------------------------------------------------------
img_gn = perturbed["gaussian_noise"]
mask_gn = sector_masks["gaussian_noise"]

# --- Parameter estimation ---------------------------------------------------
# Inside a dark blood-pool patch the clean signal is ~0, so the ROI std is a
# direct empirical estimate of the additive Gaussian noise sigma. We use it to
# tie the bilateral filter's intensity tolerance to the noise level.
sigma_est = float(img_gn[rr, cc].std())
print(f"Estimated noise sigma (from dark ROI): {sigma_est:.4f}")

# --- Filter -----------------------------------------------------------------
# Bilateral filter = spatial Gaussian * intensity Gaussian. For each pixel it
# averages neighbours weighted both by their distance and by how close their
# intensity is to the centre pixel — so pixels *across an edge* are down-
# weighted and the edge is preserved.
# Knobs:
#   * sigma_spatial : neighbourhood radius (px). Larger -> more smoothing.
#   * sigma_color   : intensity tolerance. If it is much smaller than the
#                     noise sigma the filter under-smooths; tying it to
#                     `sigma_est` keeps the ratio consistent across frames.
bilateral_result = restoration.denoise_bilateral(
    img_gn,
    sigma_color=1.5 * sigma_est,
    sigma_spatial=3.0,
    channel_axis=None,
).astype(np.float32)

# --- Mask restore -----------------------------------------------------------
bilateral_result = np.where(mask_gn, bilateral_result, 0.0)

show_comparison(
    img_gn,
    bilateral_result,
    "gaussian_noise (perturbed)",
    f"bilateral (sc=1.5*{sigma_est:.3f}, ss=3)",
    "2.2 — Additive Gaussian denoising via bilateral filtering",
)

## 2.3 `speckle` → Non-local means (NLM)

**Model.** Multiplicative-like signal-dependent noise `y = x + sqrt(x) * n`, `n ~ N(0, σ²)`. The variance depends on the local mean, so a filter with a single global variance (like bilateral) would either over- or under-smooth.

**Filter.** NLM denoises each pixel by a weighted average of *all* pixels in the frame whose *neighbourhood texture* is similar to its own:

$$\hat{u}(x) = \frac{1}{Z(x)} \sum_{y \in \Omega} w(x, y)\, u(y),\quad w(x, y) = \exp\!\left(-\frac{\|P_x - P_y\|^2}{h^2}\right)$$

Because cardiac walls have repetitive fibrous texture, many patches across the frame share the same "true" signal — averaging over them cancels the multiplicative noise without blurring the anatomy.

**Main knobs.**
- `h` — filter strength, tied here to the estimated noise sigma.
- `patch_size` — size of the local neighbourhood used for the similarity comparison.
- `patch_distance` — half-width of the search window.

> NLM is computationally heavier than the local filters above; `fast_mode=True` uses a single-channel estimator that is significantly faster with minimal quality loss.

In [ ]:
# --- Input ------------------------------------------------------------------
img_sk = perturbed["speckle"]
mask_sk = sector_masks["speckle"]

# --- Parameter estimation ---------------------------------------------------
# Speckle variance scales with the local mean, so a single global sigma is
# only a rough calibration point. We use the *bright* ROI std because that is
# where the multiplicative noise is loudest; it gives a good upper bound for
# the filter strength `h`.
sigma_sk = float(img_sk[brr, bcc].std())
print(f"Speckle sigma estimate (bright ROI): {sigma_sk:.4f}")

# --- Filter -----------------------------------------------------------------
# Non-local means denoises each pixel by a weighted average of *all* pixels
# whose surrounding patch is similar to the current patch — exploiting the
# fact that cardiac walls have repetitive fibrous texture, so many patches
# across the frame share the same underlying signal.
# Knobs:
#   * h              : filter strength (larger -> more averaging). Tied to
#                      the noise level via `sigma_sk`.
#   * patch_size     : side of the patch used for the similarity comparison.
#   * patch_distance : half-width of the search window (px).
#   * fast_mode      : single-channel fast estimator; much faster with
#                      negligible quality loss on this problem size.
nlm_result = restoration.denoise_nl_means(
    img_sk,
    h=0.8 * sigma_sk,
    patch_size=5,
    patch_distance=6,
    fast_mode=True,
    preserve_range=True,
).astype(np.float32)

# --- Mask restore -----------------------------------------------------------
nlm_result = np.where(mask_sk, nlm_result, 0.0)

show_comparison(
    img_sk,
    nlm_result,
    "speckle (perturbed)",
    f"NLM (h=0.8*{sigma_sk:.3f})",
    "2.3 — Speckle reduction via non-local means",
)

## 2.4 `contrast` → CLAHE

**Model.** Contrast attenuation compresses the sector's dynamic range around its mean: `y = mu + s * (x**gamma - mu)`, with `s < 1`.

**Filter.** **Contrast-Limited Adaptive Histogram Equalisation (CLAHE)** re-expands the dynamic range tile-by-tile:

- The frame is split into non-overlapping tiles.
- Within each tile the histogram is equalised, but capped at `clip_limit` before redistribution — this prevents equalisation from overshooting on nearly-uniform tiles.
- Bilinear interpolation between tile transforms avoids tile-boundary artefacts.

**Main knobs.**
- `clip_limit` — higher → more aggressive contrast recovery, more noise amplification.
- `kernel_size` — smaller tiles capture finer local structure but can create artefacts.

In [ ]:
# --- Input ------------------------------------------------------------------
img_c = perturbed["contrast"]
mask_c = sector_masks["contrast"]

# --- Filter -----------------------------------------------------------------
# CLAHE (Contrast-Limited Adaptive Histogram Equalisation) runs histogram
# equalisation *tile-by-tile* rather than globally, so local contrast is
# restored without darkening one region to brighten another. Bilinear
# interpolation between tile transforms hides the tile boundaries.
# Knobs:
#   * kernel_size : side of each tile in pixels. Smaller tiles capture finer
#                   local structure but can create block artefacts.
#   * clip_limit  : caps the histogram before redistribution to prevent
#                   runaway noise amplification on near-uniform tiles.
#   * nbins       : number of histogram bins; 256 matches 8-bit intensity
#                   resolution and is a safe default.
clahe_result = exposure.equalize_adapthist(
    img_c,
    kernel_size=64,
    clip_limit=0.02,
    nbins=256,
).astype(np.float32)

# --- Mask restore -----------------------------------------------------------
clahe_result = np.where(mask_c, clahe_result, 0.0)

show_comparison(
    img_c,
    clahe_result,
    "contrast (perturbed)",
    "CLAHE (clip=0.02, tile=64)",
    "2.4 — Contrast recovery via CLAHE",
)

### Verifying the fix in histogram space

The image panels above make CLAHE look like a subtle change — but its whole point is to **redistribute intensities**, and that is most visible in the histogram, not in the image. Below we plot the whole-sector histogram of the clean reference, the contrast-perturbed frame, and the CLAHE'd output side by side. We also re-compute the two statistics we used in Section 1.4 to *diagnose* contrast attenuation (`sector_std` and `P95 − P5`), so we can numerically check that CLAHE has pushed them back up.

**What to look for:**
- The perturbed sector histogram is compressed into a narrow band around the sector mean (this is exactly the §1.4 fingerprint that triggered the `contrast?` verdict).
- After CLAHE the distribution is re-spread across `[0, 1]` — but *flatter* than the clean reference, not identical to it. CLAHE maximises local contrast subject to the clip limit; it does not attempt to invert the specific `y = mu + s * (x**gamma - mu)` transform. That is the honest limitation of a reference-free classical filter.

In [ ]:
# Whole-sector pixel arrays for the three frames we want to compare.
clean_sector = clean[clean_mask]
pert_sector = img_c[mask_c]
clahe_sector = clahe_result[mask_c]


def _spread(values: np.ndarray) -> tuple[float, float]:
    """Return the two reference-free spread statistics used in §1.4.

    Parameters
    ----------
    values : np.ndarray
        1-D array of pixel intensities from a single frame (usually a
        sector-masked ravel).

    Returns
    -------
    tuple of float
        ``(std, P95 - P5)`` — standard deviation and 5th-to-95th-percentile
        range. Both shrink under contrast attenuation and stay put under any
        other kernel.
    """
    p5, p95 = np.percentile(values, [5, 95])
    return float(values.std()), float(p95 - p5)


# Numerical check: both statistics should recover from the perturbed values
# back towards (but not exactly to) the clean baseline.
print(f"{'frame':>22s} | sector std | P95 - P5")
print("-" * 50)
for label, arr in [
    ("clean (reference)", clean_sector),
    ("contrast (perturbed)", pert_sector),
    ("CLAHE'd", clahe_sector),
]:
    std, ptp = _spread(arr)
    print(f"{label:>22s} | {std:10.3f} | {ptp:8.3f}")

# Side-by-side histograms — shared bin edges and y-axis so the shape
# differences are directly comparable.
edges = np.linspace(0.0, 1.0, 65)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True, sharey=True)

axes[0].hist(clean_sector.ravel(), bins=edges, color="lightgrey")
axes[0].set_title("Clean (reference)", fontsize=10)
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Count (sector only)")

axes[1].hist(pert_sector.ravel(), bins=edges, color="tab:orange", alpha=0.85)
axes[1].set_title("Contrast (perturbed) — compressed", fontsize=10)
axes[1].set_xlabel("Intensity")

axes[2].hist(clahe_sector.ravel(), bins=edges, color="tab:green", alpha=0.85)
axes[2].set_title("CLAHE'd — re-spread (flatter than clean)", fontsize=10)
axes[2].set_xlabel("Intensity")

for ax in axes:
    ax.set_xlim(0, 1)

fig.suptitle(
    "2.4 — CLAHE closes the §1.4 loop: compressed distribution → re-spread distribution",
    fontsize=11,
    fontweight="bold",
)
plt.show()

## 2.5 `gaussian_blur` → unsharp mask

**Model.** Blur = convolution with a smoothing kernel that averages neighbouring pixels — high spatial frequencies are attenuated, so fine edges (endocardial border, papillary muscles) look washed out.

**Filter.** **Unsharp masking** (Gonzalez & Woods §3.6.3) sharpens by *adding back* a scaled high-pass of the image:

$$\hat{u}(x, y) = u(x, y) + \alpha \bigl( u(x, y) - G_\sigma * u(x, y) \bigr)$$

Reading the formula step by step: $G_\sigma * u$ is a Gaussian-blurred copy of $u$; subtracting it from $u$ leaves the high-frequency content (edges and texture); multiplying by $\alpha$ boosts that content; adding it back to $u$ produces a sharper image.

**Main knobs.**
- `radius` — sigma of the internal Gaussian low-pass. Should roughly match the *scale* of the blur you want to counter (too small → no effect; too large → halos around bright edges).
- `amount` — the multiplier $\alpha$. Larger → more sharpening, but also amplifies any residual noise. Typical values are `0.5`–`2.0`.

> ⚠ Unsharp masking amplifies noise together with edges. If the frame also carries additive/speckle noise, denoise first (§2.2 / §2.3), *then* sharpen.

In [ ]:
# --- Input ------------------------------------------------------------------
img_b = perturbed["gaussian_blur"]
mask_b = sector_masks["gaussian_blur"]

# --- Filter -----------------------------------------------------------------
# Unsharp masking computes  img + amount * (img - GaussianBlur(img, radius)).
# The parenthesised difference is the high-pass component (edges and texture);
# the multiplier controls how strongly it is added back to the image.
# Knobs:
#   * radius : sigma (in pixels) of the internal Gaussian low-pass. We pick
#              1.2 to match the scale of the moderate blur applied in
#              noise_generation.ipynb (sigmas ~ 1.0-1.2).
#   * amount : sharpening strength. 1.5 is a moderate value that restores
#              edge crispness without producing obvious halos.
unsharp_result = filters.unsharp_mask(
    img_b.astype(np.float32),
    radius=1.2,
    amount=1.5,
).astype(np.float32)

# --- Mask restore -----------------------------------------------------------
# Clip to [0, 1] before restoring the black background — the high-pass
# addition can push a few edge pixels slightly outside the display range.
unsharp_result = np.clip(unsharp_result, 0.0, 1.0)
unsharp_result = np.where(mask_b, unsharp_result, 0.0)

show_comparison(
    img_b,
    unsharp_result,
    "gaussian_blur (perturbed)",
    "unsharp mask (radius=1.2, amount=1.5)",
    "2.5 — Blur mitigation via unsharp masking",
)

# --- Verification: did the sharpening recover edge energy? ------------------
# §1.4 loop closure — the dark-ROI std that *detected* the blur should climb
# back toward the clean value once unsharp masking has re-injected the
# high-frequency content.
print("\nVerification — §1.4 dark-ROI std after unsharp masking")
print(f"{'frame':>22s} | dark ROI std")
print("-" * 40)
for label, arr in [
    ("clean (reference)", clean),
    ("blurred (perturbed)", img_b),
    ("unsharp'd", unsharp_result),
]:
    roi_std = float(arr[rr, cc].std())
    print(f"{label:>22s} | {roi_std:12.4f}")

---

# Summary — identification × mitigation

The grid below shows all five mitigation outputs alongside the clean reference. Ideally each panel should visibly move *back* towards the top-left clean panel — that's the visual signature of a correctly identified noise + correctly chosen filter.

In [ ]:
results = [
    ("Clean (reference)", clean),
    ("salt_pepper → median", median_result),
    ("gaussian_noise → bilateral", bilateral_result),
    ("speckle → NLM", nlm_result),
    ("contrast → CLAHE", clahe_result),
    ("gaussian_blur → unsharp mask", unsharp_result),
]

# 2x3 grid — top-left is the clean baseline; every other panel should
# visually move *back* towards it once the correct filter is applied.
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
for ax, (title, img) in zip(axes.ravel(), results):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.set_title(title, fontsize=10)
    ax.axis("off")

fig.suptitle(
    "Identification → mitigation: one filter per noise type",
    fontsize=12,
    fontweight="bold",
)
plt.show()

## When to use each filter

| Filter | Primary use case | Main caveat |
|---|---|---|
| **Median (3×3)** | Isolated impulse noise (salt-and-pepper) | Slightly blurs fine anatomy at larger sizes |
| **Bilateral** | Additive Gaussian noise while preserving edges | Slower than a plain Gaussian blur; `sigma_color` needs tuning |
| **NLM** | Multiplicative / speckle noise | Computationally heavy; `h` must be tied to the noise level |
| **CLAHE** | Contrast attenuation / illumination flattening | Can amplify residual noise if `clip_limit` is too high |
| **Unsharp mask** | Mild Gaussian blur / edge crispness recovery | Amplifies any residual noise together with the edges — denoise first |

> **Practical chain for a mixed-noise frame.** In lab-0 the released frames typically stack several kernels. A workable pipeline is: **median (remove S&P) → NLM (attenuate speckle) → bilateral (residual AWGN) → CLAHE (restore contrast) → unsharp mask (recover edge crispness last, once the noise floor is down)**. Skip any stage whose noise you didn't detect in Section 1.